# Lab 10 · Đảm bảo chất lượng chéo bảng

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 10**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo bài 10 làm sạch bảng `listings`. Lab này kiểm tra tính nhất quán giữa
hai bảng thông qua khoá ngoại, khoá tự nhiên và các cột dẫn xuất có sẵn.

*Từ tuần này phần kỹ năng kéo dài khoảng 50 phút; 30 phút cuối dành cho hoạt động hỗ trợ bài tập lớn.*

**Mục tiêu bài lab**

1. Kiểm tra miền thời gian và khoá ngoại giữa hai bảng, rồi diễn giải đúng kết quả của phép kiểm.
2. Chứng minh một bảng **không có khoá tự nhiên** và nói được hệ quả.
3. Đối chiếu cột dẫn xuất có sẵn với con số tự đếm và phân tích phần chênh lệch.
4. Đóng gói mọi phép kiểm thành `qa_report` ghi ra file.

Phiên bản đề: `qa-cross-table-v1` (25/09/2026). Lưu đúng tên **`lab-10.ipynb`**.

## Cách làm và nộp bài

- Các bài có hướng dẫn: nên **tự gõ, không dùng AI** — các bài kiểm tra định kỳ 🚫 đóng ở giờ lý thuyết
  đánh giá các kỹ năng này. Bài tự làm cuối lab được gắn nhãn ✅ mở: được dùng AI, kèm trách nhiệm khai báo.
- Điền thân hàm TODO và phần trả lời Markdown. Mỗi phép kiểm là **một hàm nhận bảng qua tham số**,
  để cùng bộ quy tắc chạy được trên mốc chụp khác (đúng yêu cầu của bài tập lớn).
- Giữ tên hàm, tham số, cell ID và metadata; viết helper/import thêm trong chính cell bài làm.
  Grader chỉ chạy cell câu đó với bảng riêng (mốc chụp khác, có review mồ côi, có cột lệch…),
  không phụ thuộc biến `ds`, `rv`, `MOC` ở cell khác. `pd` và `np` được cung cấp sẵn khi chấm.
- Public checks chỉ là ví dụ tự kiểm tra trên dữ liệu giả lập. Một check lỗi không dừng các check sau.
  TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- Phép kiểm **chỉ đếm và báo cáo**: không sửa, không xoá dòng của bảng đầu vào; không ghi cứng kết quả.
- Lab chạy với hành vi **Copy-on-Write** của pandas 3 (bật sẵn trong cell dữ liệu và khi chấm).
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1, 2, 3 — mỗi bài 10 điểm | 30 | Tự động |
| Bài 4 (đối chiếu tổng review) 15 · Bài 5 (lệch LTM) 20 · Bài 6 (`qa_report`) 15 | 50 | Tự động |
| Diễn giải: chính sách mốc, số 0 vi phạm, khoá tự nhiên, phân bố độ lệch | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Phần 🧭 hỗ trợ bài tập lớn ở cuối lab làm theo nhóm, không tính vào điểm lab.

In [10]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

## Dữ liệu dùng trong lab

Mặc định `USE_SNAPSHOT = False`: notebook dùng **dữ liệu giả lập** (6 chỗ ở, 18 đánh giá) có cùng các hiện tượng
như snapshot thật — vài đánh giá sau mốc chụp, cặp `(listing_id, date)` lặp lại, cột LTM lệch −1.

Đặt `USE_SNAPSHOT = True` để đọc bản **visualisations** (`listings.csv`, `reviews.csv`) của snapshot Santiago **2026-06-29**.
Hằng `MOC` là mốc chụp công bố; `LTM_BAT_DAU` là ngày bắt đầu cửa sổ 12 tháng thử nghiệm. Public checks luôn dùng dữ liệu giả lập.

In [11]:
import numpy as np
import pandas as pd

# Hành vi pandas 3 (Copy-on-Write).
if pd.__version__.startswith("2."):
    pd.set_option("mode.copy_on_write", True)

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations"
MOC = "2026-06-29"
LTM_BAT_DAU = "2025-06-30"

# Dữ liệu GIẢ LẬP: 6 chỗ ở và 18 đánh giá, cùng cấu trúc bản visualisations của Inside Airbnb.
DEMO_RV = pd.DataFrame({
    "listing_id": [11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 13, 13, 13, 13, 14, 14, 16],
    "date": pd.to_datetime([
        "2019-03-01", "2024-12-01", "2025-06-30", "2025-08-10", "2025-08-10", "2026-06-29", "2026-06-30",
        "2010-11-13", "2025-06-30", "2026-01-05", "2026-07-01",
        "2023-04-04", "2023-04-04", "2023-04-04", "2025-07-01",
        "2025-12-24", "2026-06-30", "2021-02-02"]),
})
DEMO_DS = pd.DataFrame({
    "id": [11, 12, 13, 14, 15, 16],
    "name": ["Depto Lastarria", "Pieza Bellavista", "Loft Providencia", "Casa Ñuñoa", "Studio nuevo", "Hab Centro"],
    "number_of_reviews": [7, 4, 4, 2, 0, 1],
    "number_of_reviews_ltm": [4, 2, 1, 2, 0, 0],
    "last_review": pd.to_datetime(["2026-06-30", "2026-07-01", "2025-07-01", "2026-06-30", None, "2021-02-02"]),
})

# Đặt USE_SNAPSHOT = True để làm trên snapshot Santiago thật.
USE_SNAPSHOT = False

if USE_SNAPSHOT:
    ds = pd.read_csv(f"{BASE}/listings.csv", parse_dates=["last_review"])
    rv = pd.read_csv(f"{BASE}/reviews.csv", parse_dates=["date"])
    DATA_LABEL = "Snapshot Santiago 2026-06-29"
else:
    ds, rv = DEMO_DS.copy(), DEMO_RV.copy()
    DATA_LABEL = "GIẢ LẬP — 6 chỗ ở, 18 đánh giá"

print(DATA_LABEL, "·", len(ds), "chỗ ở ·", len(rv), "đánh giá")

GIẢ LẬP — 6 chỗ ở, 18 đánh giá · 6 chỗ ở · 18 đánh giá


## Bài 1 · Miền thời gian của bảng đánh giá

Tên thư mục ghi mốc chụp **29/06/2026**. Các đánh giá có ngày sau mốc đó cần được gắn cờ.

### Hợp đồng hàm · 10 điểm tự động

`time_domain(rv: pd.DataFrame, moc: str) -> dict`

Nhận bảng đánh giá (cột `date` kiểu `datetime64`, ít nhất một dòng) và mốc `"YYYY-MM-DD"`. Trả dict có đúng các khóa:
`ngay_dau`, `ngay_cuoi` (`Timestamp` sớm nhất/muộn nhất của `date`) và `so_sau_moc` (int, số dòng có `date > moc`).
Không sửa input. Đối chiếu snapshot: từ **2010-11-13** đến **2026-07-01**; **204** dòng sau mốc chụp.

In [12]:
def time_domain(rv: pd.DataFrame, moc: str) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    moc_dt = pd.to_datetime(moc)
    return {
        "ngay_dau" : rv["date"].min(),
        "ngay_cuoi" : rv["date"].max(),
        "so_sau_moc" : int((rv["date"] > moc_dt).sum())
    }

In [13]:
def check_q1():
    result = time_domain(DEMO_RV, MOC)
    assert set(result) == {"ngay_dau", "ngay_cuoi", "so_sau_moc"}
    assert pd.Timestamp(result["ngay_dau"]) == pd.Timestamp("2010-11-13")
    assert pd.Timestamp(result["ngay_cuoi"]) == pd.Timestamp("2026-07-01")
    assert result["so_sau_moc"] == 3, "30/06 (2 dòng) và 01/07 (1 dòng) nằm sau mốc 29/06"
    assert time_domain(DEMO_RV, "2026-07-01")["so_sau_moc"] == 0

public_check("Q1 · ví dụ công khai", check_q1)

[ĐẠT] Q1 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


**Chính sách mốc chụp:** 

1. Các dòng sau mốc rơi vào những ngày nào?

Mốc chụp theo tên thư mục là 29/06/2026 (2026-06-29), trong khi ngày muộn nhất ghi nhận trong bảng là 01/07/2026 (2026-07-01). Vì điều kiện sau mốc là date > moc, toàn bộ các dòng sau mốc chỉ rơi vào 2 ngày: 30/06/2026 (2026-06-30) và 01/07/2026 (2026-07-01). Trong tập dữ liệu thực tế có tổng cộng 204 dòng thuộc hai ngày này (riêng bộ kiểm thử công khai DEMO_RV có 3 dòng: 2 dòng ngày 30/06/2026 và 1 dòng ngày 01/07/2026).

2. Có nên đổi mốc công bố thành ngày muộn nhất trong tệp (01/07/2026) không? Vì sao

Không nên đổi mốc công bố thành 01/07/2026, vì các lý do sau:Bảo toàn tính nguyên vẹn và nguồn gốc dữ liệu (Data Provenance): Mốc 29/06/2026 phản ánh thời điểm chốt dữ liệu theo quy ước phát hành. Việc phát sinh thêm các bản ghi vào ngày 30/06 và 01/07 thường do chênh lệch múi giờ (ví dụ: giờ UTC so với giờ địa phương của máy chủ) hoặc quá trình trích xuất dữ liệu kéo dài qua nửa đêm. Tự ý sửa mốc công bố chỉ để "khớp" với dữ liệu sẽ che giấu sự bất nhất thực tế của bộ dữ liệu.   Dữ liệu ngày cuối cùng không trọn vẹn (Right-censored data): Ngày 01/07/2026 chỉ là thời điểm tiến trình thu thập dừng lại giữa chừng, không đại diện cho đủ 24 giờ của ngày đó. Nếu coi 01/07/2026 là mốc hoàn chỉnh, các phân tích xu hướng theo ngày sẽ ghi nhận mức sụt giảm giả tạo ở ngày cuối cùng.Đảm bảo tính nhất quán và tránh rò rỉ dữ liệu (Data Leakage): Khi nhiều người hoặc nhiều mô hình cùng khai thác bộ dữ liệu này dựa trên mốc chuẩn 29/06/2026 để chia tập huấn luyện/kiểm thử theo thời gian, việc tự ý dời mốc lên 01/07/2026 sẽ làm sai lệch tập mẫu và mất khả năng tái lập kết quả.Tuân thủ nguyên tắc gắn cờ (Flagging) thay vì sửa metadata: Giải pháp chuẩn trong kỹ thuật dữ liệu là giữ nguyên mốc 29/06/2026 và gắn cờ cho 204 dòng vượt mốc, giúp người dùng sau đó chủ động chọn lọc hoặc giữ lại tùy theo mục đích phân tích cụ thể.

## Bài 2 · Khoá ngoại: đánh giá không khớp chỗ ở

Mọi `listing_id` trong bảng đánh giá phải tồn tại trong `ds["id"]`. Nếu không, hai bảng không bảo toàn khoá ngoại.

### Hợp đồng hàm · 10 điểm tự động

`orphan_reviews(rv: pd.DataFrame, ds: pd.DataFrame) -> int`

Nhận bảng đánh giá (cột `listing_id`) và bảng chỗ ở (cột `id`). Trả **int**: số dòng đánh giá có `listing_id`
**không** nằm trong `ds["id"]` (dùng `isin` và `~`). Mỗi dòng mồ côi đều được đếm, kể cả khi cùng một `listing_id`
lặp lại. Không sửa input. Đối chiếu snapshot: **0** — khoá ngoại toàn vẹn.

In [14]:
def orphan_reviews(rv: pd.DataFrame, ds: pd.DataFrame) -> int:
    # TODO: triển khai đúng hợp đồng ở trên.

    return int((~rv["listing_id"].isin(ds["id"])).sum())

In [15]:
def check_q2():
    result = orphan_reviews(DEMO_RV, DEMO_DS)
    assert isinstance(result, (int, np.integer)) and not isinstance(result, bool)
    assert result == 0
    thieu = DEMO_DS[DEMO_DS["id"] != 11]
    assert orphan_reviews(DEMO_RV, thieu) == 7, "Bỏ chỗ ở 11 thì 7 đánh giá của nó thành mồ côi"

public_check("Q2 · ví dụ công khai", check_q2)

[ĐẠT] Q2 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


**Số 0 vi phạm:** vì sao phép kiểm cho kết quả 0 vẫn phải ghi trong báo cáo? 

Phép kiểm cho kết quả 0 vi phạm vẫn phải ghi vào báo cáo vì: (1) Xác nhận minh bạch rằng ràng buộc toàn vẹn khoá ngoại đã được kiểm tra và đạt chuẩn, tránh nhầm lẫn với việc bỏ sót bước kiểm tra; (2) Làm cơ sở đảm bảo an toàn cho các phép nối bảng (JOIN) phía sau không bị thất thoát dữ liệu hay sinh giá trị NULL bất thường; và (3) Thiết lập mốc chuẩn (baseline) để dễ dàng phát hiện bất thường nếu dữ liệu ở các kỳ cập nhật (snapshot) sau phát sinh lỗi $> 0$.

## Bài 3 · Khoá tự nhiên: bảng đánh giá rút gọn có khoá không?

### Hợp đồng hàm · 10 điểm tự động

`duplicate_pairs(rv: pd.DataFrame, cot: list = ["listing_id", "date"]) -> int`

Nhận bảng và danh sách cột ứng viên làm khoá. Trả **int**: số dòng lặp lại theo các cột đó
(`duplicated(subset=cot)` — lần xuất hiện đầu tiên không tính). Chỉ đếm, **không** `drop_duplicates`, không sửa input.
Đối chiếu snapshot: **3.548** dòng "trùng" theo `(listing_id, date)`.

In [16]:
def duplicate_pairs(rv: pd.DataFrame, cot: list = ["listing_id", "date"]) -> int:
    # TODO: triển khai đúng hợp đồng ở trên.

    return int(rv.duplicated(subset=cot).sum())

In [17]:
def check_q3():
    result = duplicate_pairs(DEMO_RV)
    assert isinstance(result, (int, np.integer)) and not isinstance(result, bool)
    assert result == 3, "Chỗ ở 11 ngày 10/08/2025 (1 dòng lặp) và chỗ ở 13 ngày 04/04/2023 (2 dòng lặp)"
    assert duplicate_pairs(DEMO_RV, ["listing_id"]) == 13
    assert len(DEMO_RV) == 18
public_check("Q3 · ví dụ công khai", check_q3)

[ĐẠT] Q3 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


**Khoá tự nhiên:** 

1. Các dòng "trùng" theo (listing_id, date) có phải lỗi không?

Không phải lỗi. Phần lớn các dòng này là những đánh giá hoàn toàn hợp lệ trong thực tế, xuất phát từ hai nguyên nhân chính:

Đặc thù nghiệp vụ lưu trú: Một mã chỗ ở (listing_id) có thể đại diện cho một cơ sở nhiều phòng (homestay, căn hộ dịch vụ, boutique hotel) hoặc tiếp nhận nhiều lượt khách trả phòng trong cùng một ngày. Việc nhiều khách hàng khác nhau cùng viết đánh giá cho cùng một chỗ ở trong một ngày (date) là hiện tượng bình thường.

Hạn chế của bảng đánh giá rút gọn: Bảng rút gọn chỉ giữ lại hai cột listing_id và date, đã lược bỏ khóa chính thực sự của từng lượt đánh giá (như review_id), mã người dùng (reviewer_id) và nội dung bình luận (comments). Do đó, cặp (listing_id, date) không phải là khóa tự nhiên (natural key) của bảng; 3.548 dòng lặp lại theo cặp cột này chỉ cho thấy các đánh giá xảy ra cùng ngày tại cùng chỗ ở chứ không chứng minh dữ liệu bị lỗi trùng lặp kỹ thuật.

2. Nếu chạy drop_duplicates(subset=["listing_id", "date"]) thì sao?Việc xóa trùng theo (listing_id, date) sẽ gây ra mất mát dữ liệu hợp lệ (data loss) và làm sai lệch nghiêm trọng các phân tích về sau:Hụt dữ liệu có hệ thống: Lệnh này sẽ giữ lại bản ghi đầu tiên và xóa bỏ toàn bộ 3.548 đánh giá hợp lệ còn lại rơi vào cùng ngày tại cùng chỗ ở.   Đánh giá thấp mức độ phổ biến của các chỗ ở lớn: Những chỗ ở đông khách, nhiều phòng hoặc hoạt động hiệu quả nhất chính là nơi có xác suất nhận nhiều đánh giá trong một ngày cao nhất, nên sẽ bị cắt giảm dữ liệu nhiều nhất.Làm phẳng các đỉnh mùa vụ (Seasonality distortion): Vào các dịp cuối tuần, lễ hoặc mùa cao điểm du lịch, lượng đánh giá tập trung theo ngày tăng mạnh. Xóa bớt các đánh giá trùng ngày sẽ làm sai lệch biểu đồ xu hướng theo thời gian và đánh giá thấp nhu cầu thực tế.Sai lệch các phép tổng hợp (Aggregation): Mọi chỉ số đếm tổng số đánh giá theo chỗ ở, theo khu vực hoặc theo tháng/năm đều sẽ bị thấp hơn thực tế.Chỉ nên thực hiện drop_duplicates() khi đối chiếu trên mã định danh duy nhất của từng đánh giá (review_id ở bảng đầy đủ) hoặc khi xác minh được bản ghi bị lặp nguyên dòng do lỗi thu thập dữ liệu; với bảng rút gọn chỉ có (listing_id, date), cần giữ nguyên các dòng này và chỉ đếm để nắm đặc điểm phân phối.

## Bài 4 · Đối chiếu cột dẫn xuất với con số tự đếm

Bảng `listings` có sẵn `number_of_reviews` và `number_of_reviews_ltm` (LTM, 12 tháng gần nhất) do Inside Airbnb tính.
Ta tự đếm từ bảng đánh giá để kiểm tra định nghĩa và độ nhất quán.

### Hợp đồng hàm · 15 điểm tự động

`review_count_check(rv: pd.DataFrame, ds: pd.DataFrame) -> dict`

Nhận bảng đánh giá (`listing_id`) và bảng chỗ ở (`id` không trùng, `number_of_reviews`). Trả dict có đúng các khóa:

- `dem_that`: Series int **cùng index với `ds`** — số dòng đánh giá của từng chỗ ở, đếm trên **toàn bộ** `rv`
  (`groupby("listing_id").size()` rồi `map`/`reindex` theo `ds["id"]`), chỗ ở không có đánh giá là 0;
- `so_lech_tong`: int, số chỗ ở có `number_of_reviews` khác `dem_that`.

Không thêm cột vào `ds`. Đối chiếu snapshot: **0** chỗ ở lệch trên 18.534.

In [18]:
def review_count_check(rv: pd.DataFrame, ds: pd.DataFrame) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    dem_that = ds["id"].map(rv.groupby("listing_id").size()).fillna(0)
    return {
        "dem_that" : dem_that,
        "so_lech_tong" : int((ds["number_of_reviews"] != dem_that).sum())
    }

In [19]:
def check_q4():
    before = DEMO_DS.copy()
    result = review_count_check(DEMO_RV, DEMO_DS)
    assert set(result) == {"dem_that", "so_lech_tong"}
    dem_that = result["dem_that"]
    assert isinstance(dem_that, pd.Series) and list(dem_that.index) == list(DEMO_DS.index)
    assert dem_that.tolist() == [7, 4, 4, 2, 0, 1], "Chỗ ở 15 không có đánh giá phải là 0, không phải NaN"
    assert result["so_lech_tong"] == 0
    pd.testing.assert_frame_equal(DEMO_DS, before, obj="DEMO_DS (không thêm cột)")

public_check("Q4 · ví dụ công khai", check_q4)

[ĐẠT] Q4 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


**Khớp toàn bộ nghĩa là gì?**

1. "Khớp toàn bộ" nghĩa là gì?

Trong kết quả đối chiếu (0 chỗ ở lệch trên tổng số 18.534 chỗ ở), khớp toàn bộ có nghĩa là giá trị sẵn có trong cột number_of_reviews của bảng listings trùng khớp 100% với số lượng bản ghi đếm trực tiếp từ toàn bộ bảng reviews (rv) cho từng listing_id (bao gồm cả các chỗ ở không có đánh giá nào đều mang giá trị 0). Kết quả này xác nhận hai đặc điểm của bộ dữ liệu:

Định nghĩa tính toán đồng nhất: Inside Airbnb tạo cột number_of_reviews bằng phép đếm đơn thuần toàn bộ số dòng xuất hiện của mỗi chỗ ở trong tệp đánh giá đi kèm, không lọc trùng theo ngày và không áp dụng điều kiện loại trừ nào khác.

Tính đồng bộ của snapshot: Hai bảng listings và reviews trong cùng một gói phát hành được trích xuất nhất quán từ cùng một trạng thái cơ sở dữ liệu nguồn.

2. number_of_reviews có bao gồm các đánh giá sau mốc 29/06 không?

Có, bao gồm đầy đủ. Hàm review_count_check thực hiện đếm trên toàn bộ bảng rv (bao gồm cả 204 dòng phát sinh vào ngày 30/06/2026 và 01/07/2026 sau mốc 29/06/2026) mà kết quả đối chiếu vẫn cho ra 0 chỗ ở sai lệch (so_lech_tong = 0). Điều này chứng minh Inside Airbnb đã tổng hợp cột number_of_reviews trực tiếp từ tập rv đầy đủ sau khi kết thúc quá trình thu thập dữ liệu, chứ không chốt số liệu tại ngày 29/06/2026.

3. Nếu quy trình cắt tại 29/06 thì cột dẫn xuất phải xử lý thế nào?Nếu quy trình tiền xử lý lọc bỏ các đánh giá sau mốc 29/06/2026 (rv = rv[rv["date"] <= "2026-06-29"]), việc giữ nguyên các cột dẫn xuất có sẵn trong bảng listings sẽ gây ra bất nhất dữ liệu (số đếm ở listings lớn hơn số dòng thực tế trong rv) và rò rỉ dữ liệu tương lai (data leakage). Để đảm bảo tính nhất quán theo điểm thời gian (point-in-time consistency), cần xử lý như sau:   Tính lại toàn bộ cột đếm tích lũy: Thay thế giá trị ở cột number_of_reviews bằng kết quả đếm lại trực tiếp từ bảng rv đã lọc theo điều kiện date <= "2026-06-29".Cập nhật các cột dẫn xuất theo cửa sổ thời gian: Tính toán lại number_of_reviews_ltm (12 tháng gần nhất) và reviews_per_month bằng cách neo mốc kết thúc về đúng ngày 29/06/2026 thay vì dùng con số tính đến 01/07/2026.Điều chỉnh các mốc ngày liên quan: Cập nhật lại cột last_review thành ngày đánh giá muộn nhất không vượt quá 29/06/2026 (hoặc gán giá trị rỗng NaN nếu chỗ ở đó chỉ có đánh giá phát sinh sau mốc 29/06/2026).

## Bài 5 · Cột LTM: lệch định nghĩa cửa sổ 12 tháng

Thử định nghĩa "12 tháng gần nhất" là các đánh giá có `date >= LTM_BAT_DAU` (snapshot: `"2025-06-30"`), tự đếm rồi so
với `number_of_reviews_ltm`. Xem **phân bố** độ lệch, không chỉ đếm số dòng lệch.

### Hợp đồng hàm · 20 điểm tự động

`ltm_mismatch(rv: pd.DataFrame, ds: pd.DataFrame, bat_dau: str, ket_thuc: str | None = None) -> dict`

Nhận hai bảng (ds có `id`, `number_of_reviews_ltm`), ngày bắt đầu cửa sổ và ngày kết thúc tùy chọn (`None` = không chặn
trên; có giá trị thì gồm cả ngày đó). Trả dict có đúng các khóa:

- `ltm_that`: Series int cùng index với `ds` — số đánh giá có `bat_dau <= date` (và `date <= ket_thuc` nếu có), 0 nếu không có;
- `do_lech`: Series int cùng index với `ds` — `number_of_reviews_ltm − ltm_that`;
- `so_lech_ltm`: int, số chỗ ở có `do_lech` khác 0.

Không sửa input. Đối chiếu snapshot (`bat_dau="2025-06-30"`, không chặn trên): **665** chỗ ở lệch, phần lớn lệch **−1**.

In [20]:
def ltm_mismatch(rv: pd.DataFrame, ds: pd.DataFrame, bat_dau: str, ket_thuc: str | None = None) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    dieu_kien = rv["date"] >= pd.to_datetime(bat_dau)
    if ket_thuc is not None:
        dieu_kien &= rv["date"] <= pd.to_datetime(ket_thuc)
    rv_loc = rv[dieu_kien]
    dem_theo_id = rv_loc.groupby("listing_id").size()
    ltm_that = ds["id"].map(dem_theo_id).fillna(0)
    do_lech = ds["number_of_reviews_ltm"] - ltm_that
    so_lech_ltm = int((do_lech != 0).sum())

    return {
        "ltm_that" : ltm_that,
        "do_lech" : do_lech ,
        "so_lech_ltm" : so_lech_ltm
    }

In [21]:
def check_q5():
    result = ltm_mismatch(DEMO_RV, DEMO_DS, LTM_BAT_DAU)
    assert set(result) == {"ltm_that", "do_lech", "so_lech_ltm"}
    assert result["ltm_that"].tolist() == [5, 3, 1, 2, 0, 0]
    assert result["do_lech"].tolist() == [-1, -1, 0, 0, 0, 0]
    assert result["so_lech_ltm"] == 2
    chan = ltm_mismatch(DEMO_RV, DEMO_DS, LTM_BAT_DAU, MOC)
    assert chan["ltm_that"].tolist() == [4, 2, 1, 1, 0, 0] and chan["so_lech_ltm"] == 1

public_check("Q5 · ví dụ công khai", check_q5)

[ĐẠT] Q5 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [22]:
def thuc_hanh_q1_q5():
    global so_lieu
    mien = time_domain(rv, MOC)
    ltm = ltm_mismatch(rv, ds, LTM_BAT_DAU)
    so_lieu = {
        "review_sau_moc": mien["so_sau_moc"],
        "review_mo_coi": orphan_reviews(rv, ds),
        "trung_theo_cap": duplicate_pairs(rv),
        "lech_tong_review": review_count_check(rv, ds)["so_lech_tong"],
        "lech_ltm": ltm["so_lech_ltm"],
    }
    lech = ltm["do_lech"]
    return (f"Ngày đánh giá từ {mien['ngay_dau']:%Y-%m-%d} đến {mien['ngay_cuoi']:%Y-%m-%d}\n{so_lieu}\n"
            f"Độ lệch LTM trong nhóm lệch:\n{lech[lech != 0].describe().round(2)}")

preview("Bài 1–5 trên dữ liệu đang dùng", thuc_hanh_q1_q5)

Bài 1–5 trên dữ liệu đang dùng
Ngày đánh giá từ 2010-11-13 đến 2026-07-01
{'review_sau_moc': 3, 'review_mo_coi': 0, 'trung_theo_cap': 3, 'lech_tong_review': 0, 'lech_ltm': 2}
Độ lệch LTM trong nhóm lệch:
count    2.0
mean    -1.0
std      0.0
min     -1.0
25%     -1.0
50%     -1.0
75%     -1.0
max     -1.0
dtype: float64


**Phân bố độ lệch:** 
1. Độ lệch tập trung ở giá trị nào?

Với công thức do_lech = number_of_reviews_ltm - ltm_that, phân bố độ lệch trên toàn bộ tập dữ liệu (665 chỗ ở bị lệch) tập trung áp đảo tại giá trị -1 — tương tự trên tập kiểm thử DEMO_RV, cả 2 chỗ ở lệch đều có độ lệch bằng -1.0. Điều này có nghĩa là đa số các chỗ ở bị lệch có giá trị number_of_reviews_ltm do Inside Airbnb công bố nhỏ hơn đúng 1 đánh giá so với cách tự đếm ltm_that (date >= "2025-06-30", không chặn trên). Các giá trị âm sâu hơn (-2, -3,...) chiếm tỷ lệ rất nhỏ, xảy ra khi một chỗ ở nhận nhiều hơn 1 đánh giá ngay trong ngày ranh giới bị lệch.

2. Điều đó gợi ý gì về định nghĩa cửa sổ của Inside Airbnb?

Việc độ lệch tập trung ở -1 cho thấy cửa sổ 12 tháng của Inside Airbnb hẹp hơn một chút (thường chênh đúng 1 ngày ở vùng biên) so với khoảng [2025-06-30, +∞). Cụ thể, kết quả này gợi ý hai khả năng chính về cách Inside Airbnb thiết lập ranh giới thời gian:

Lệch 1 ngày ở cận dưới (mốc bắt đầu): Inside Airbnb có thể dùng điều kiện so sánh ngặt (date > "2025-06-30") hoặc lùi đúng 365 ngày từ ngày dữ liệu muộn nhất 01/07/2026 (tức lấy từ 2025-07-01 thay vì 2025-06-30). Khi đó, những đánh giá viết đúng vào ngày 2025-06-30 bị loại khỏi number_of_reviews_ltm nhưng vẫn được tính trong ltm_that.

Áp dụng chặn trên tại ngày mốc chụp (ket_thuc = "2026-06-29"): Trong hàm ltm_mismatch, tham số ket_thuc đang để None (không chặn trên), nên ltm_that bao gồm cả các đánh giá phát sinh vào ngày 30/06 và 01/07/2026. Nếu Inside Airbnb chặn trên cửa sổ LTM tại đúng ngày mốc 2026-06-29, những chỗ ở có phát sinh đánh giá sau mốc này cũng sẽ bị lệch -1.

3. Vì sao chưa đủ để khẳng định?

Chỉ dựa vào việc phân bố do_lech tập trung ở -1 là chưa đủ cơ sở để kết luận định nghĩa chính xác của Inside Airbnb, vì những lý do sau:

Nhiều giả thuyết cùng tạo ra một mức lệch (Observational equivalence): Giá trị -1 chỉ cho biết hai cách đếm chênh nhau 1 bản ghi ở ranh giới, nhưng chưa phân biệt được bản ghi bị loại nằm ở cận dưới (2025-06-30), nằm ở cận trên (30/06 – 01/07/2026), hay phát sinh do chênh lệch múi giờ (giờ UTC của máy chủ so với giờ địa phương nơi đặt chỗ ở) khiến dấu thời gian bị dịch qua nửa đêm.

Chưa kiểm chứng đối chiếu từng tổ hợp tham số: Cần chạy thử nghiệm thay đổi tham số bat_dau (ví dụ "2025-07-01") và ket_thuc (ví dụ "2026-06-29") trong hàm ltm_mismatch để kiểm tra xem cấu hình nào đưa so_lech_ltm về 0 (hoặc gần 0 nhất).

Chưa soi trực tiếp dữ liệu ngày của nhóm lệch và năm nhuận: Cần kiểm tra trực tiếp cột date của 665 chỗ ở bị lệch xem chúng rơi vào ngày 2025-06-30 hay sau 2026-06-29, đồng thời đối chiếu thêm trên dữ liệu năm nhuận (như năm 2024) để xác định Inside Airbnb tính cửa sổ theo 365 ngày cố định hay 12 tháng dương lịch.

## Bài 6 · Đóng gói `qa_report`

Gom kết quả các phép kiểm thành một bảng ba cột và ghi ra file — sản phẩm theo yêu cầu của bài tập lớn.
Cell thực hành phía trên đã gom số liệu vào dict `so_lieu`.

### Hợp đồng hàm · 15 điểm tự động

`qa_report(so_dong: dict, ghi_chu: dict, path) -> pd.DataFrame`

Nhận dict `{tên quy tắc: số dòng (int)}` (giữ thứ tự khóa), dict `{tên quy tắc: ghi chú}` và đường dẫn file CSV
(thư mục cha đã tồn tại). Tạo DataFrame đúng ba cột `quy_tac`, `so_dong`, `ghi_chu`, **mỗi quy tắc trong `so_dong` một dòng**
theo đúng thứ tự — kể cả quy tắc có 0 dòng; quy tắc không có trong `ghi_chu` thì ghi chú là chuỗi rỗng `""`.
Ghi bằng `to_csv(path, index=False)` (UTF-8), đọc lại bằng `pd.read_csv(path, keep_default_na=False)` và trả bảng đọc lại.
Không sửa input.

In [23]:
def qa_report(so_dong: dict, ghi_chu: dict, path) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    quy_tac_list = list(so_dong.keys())
    so_dong_list = list(so_dong.values())
    ghi_chu = [ghi_chu.get(k,"") for k in quy_tac_list]
    df = pd.DataFrame(
        {
            "quy_tac" : quy_tac_list,
            "so_dong" : so_dong_list,
            "ghi_chu" : ghi_chu
        }
    )
    df.to_csv(path, index = False, encoding = "utf-8")
    return pd.read_csv(path, keep_default_na= False)

In [24]:
def check_q6():
    import tempfile
    from pathlib import Path
    so = {"review_sau_moc": 3, "review_mo_coi": 0, "trung_theo_cap": 3}
    ghi = {"review_sau_moc": "date > mốc công bố", "trung_theo_cap": "không xoá"}
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "qa_report.csv"
        kq = qa_report(so, ghi, path)
        assert path.exists(), "Cần thực sự ghi file"
        header = path.read_text(encoding="utf-8").splitlines()[0]
    assert header == "quy_tac,so_dong,ghi_chu", "Kiểm tra index=False và thứ tự cột"
    assert isinstance(kq, pd.DataFrame) and kq.shape == (3, 3)
    assert kq["quy_tac"].tolist() == list(so), "Giữ thứ tự quy tắc, kể cả quy tắc có 0 dòng"
    assert kq.set_index("quy_tac")["so_dong"].to_dict() == so
    assert kq.loc[1, "ghi_chu"] == ""

public_check("Q6 · ví dụ công khai", check_q6)

[ĐẠT] Q6 · ví dụ công khai: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [25]:
GHI_CHU = {
    "review_sau_moc": "date > mốc công bố; áp dụng chính sách cắt mốc đã ghi rõ",
    "review_mo_coi": "listing_id không có trong listings",
    "trung_theo_cap": "không xoá; hai cột không phải khoá tự nhiên",
    "lech_tong_review": "cột dẫn xuất so với số tự đếm",
    "lech_ltm": "khác định nghĩa mốc 12 tháng",
}
preview("Bài 6 — qa_report_reviews.csv", lambda: qa_report(so_lieu, GHI_CHU, "qa_report_reviews.csv"))

Bài 6 — qa_report_reviews.csv
            quy_tac  so_dong  \
0    review_sau_moc        3   
1     review_mo_coi        0   
2    trung_theo_cap        3   
3  lech_tong_review        0   
4          lech_ltm        2   

                                             ghi_chu  
0  date > mốc công bố; áp dụng chính sách cắt mốc...  
1                 listing_id không có trong listings  
2        không xoá; hai cột không phải khoá tự nhiên  
3                      cột dẫn xuất so với số tự đếm  
4                       khác định nghĩa mốc 12 tháng  


## Bài tự làm ✅ mở — không cộng vào 80 điểm tự động

**Bộ QA trên mốc chụp khác.** Chạy lại năm phép kiểm trên mốc **2025-09-27**
(đổi `BASE` thành `.../2025-09-27/visualisations`; cập nhật `MOC` và cửa sổ LTM).
Quy tắc nào cho kết quả khác? Ghi lại thay đổi cần thiết để cùng một bộ quy tắc chạy được
trên một mốc chụp khác (bộ test chung khi chấm có thể dùng mốc như vậy).

In [26]:
import re
import pandas as pd

def chay_bo_qa(base_url: str, moc: str) -> dict:
    """Chạy lại toàn bộ 5 phép kiểm QA trên một mốc chụp bất kỳ."""
    # 1. Đọc dữ liệu từ thư mục visualisations của mốc tương ứng
    ds_moi = pd.read_csv(f"{base_url}/listings.csv")
    rv_moi = pd.read_csv(f"{base_url}/reviews.csv")
    rv_moi["date"] = pd.to_datetime(rv_moi["date"])

    # 2. Tính tự động cận dưới cửa sổ LTM (12 tháng tính đến mốc chụp)
    # Ví dụ: moc = "2026-06-29" -> "2025-06-30"; moc = "2025-09-27" -> "2024-09-28"
    ltm_bat_dau = (
        pd.Timestamp(moc) - pd.DateOffset(years=1) + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")

    # 3. Chạy 5 phép kiểm (Q1 - Q5)
    mien = time_domain(rv_moi, moc)
    ltm = ltm_mismatch(rv_moi, ds_moi, ltm_bat_dau)

    ket_qua = {
        "moc": moc,
        "ltm_bat_dau": ltm_bat_dau,
        "ngay_dau": mien["ngay_dau"].strftime("%Y-%m-%d"),
        "ngay_cuoi": mien["ngay_cuoi"].strftime("%Y-%m-%d"),
        "review_sau_moc": mien["so_sau_moc"],
        "review_mo_coi": orphan_reviews(rv_moi, ds_moi),
        "trung_theo_cap": duplicate_pairs(rv_moi),
        "lech_tong_review": review_count_check(rv_moi, ds_moi)["so_lech_tong"],
        "lech_ltm": ltm["so_lech_ltm"],
    }
    return ket_qua, ltm["do_lech"]

# Cập nhật BASE sang mốc 2025-09-27 (giữ nguyên đường dẫn thành phố hiện tại)
MOC_2025 = "2025-09-27"
BASE_2025 = re.sub(r"\d{4}-\d{2}-\d{2}", MOC_2025, BASE)

kq_2025, do_lech_2025 = chay_bo_qa(BASE_2025, MOC_2025)

print("Kết quả QA trên mốc 2025-09-27:")
for k, v in kq_2025.items():
    print(f"  - {k}: {v}")

print("\nPhân bố độ lệch LTM (nhóm lệch != 0):")
print(do_lech_2025[do_lech_2025 != 0].value_counts().sort_index())

# So sánh nhanh với mốc gốc (nếu biến so_lieu từ cell trước đã tồn tại)
if "so_lieu" in globals():
    so_sanh = pd.DataFrame({
        "Moc_2026_06_29": so_lieu,
        "Moc_2025_09_27": {k: kq_2025[k] for k in so_lieu}
    })
    print("\nBảng đối chiếu 5 quy tắc giữa hai mốc:")
    print(so_sanh)

Kết quả QA trên mốc 2025-09-27:
  - moc: 2025-09-27
  - ltm_bat_dau: 2024-09-28
  - ngay_dau: 2010-11-13
  - ngay_cuoi: 2025-09-27
  - review_sau_moc: 0
  - review_mo_coi: 0
  - trung_theo_cap: 2378
  - lech_tong_review: 0
  - lech_ltm: 385

Phân bố độ lệch LTM (nhóm lệch != 0):
1.0    385
Name: count, dtype: int64

Bảng đối chiếu 5 quy tắc giữa hai mốc:
                  Moc_2026_06_29  Moc_2025_09_27
review_sau_moc                 3               0
review_mo_coi                  0               0
trung_theo_cap                 3            2378
lech_tong_review               0               0
lech_ltm                       2             385


**Bài tự làm của tôi — mốc 2025-09-27:**

Hàm `chay_bo_qa` nhận URL và mốc chụp, đổi `date` sang datetime và tính ngày đầu cửa sổ LTM bằng cách lùi một năm rồi cộng một ngày: 27/09/2025 cho 28/09/2024. Output đã lưu cho miền review từ 13/11/2010 đến 27/09/2025; không có review sau mốc hoặc review mồ côi, có 2.378 cặp listing–ngày trùng, không lệch tổng review và có 385 listing lệch LTM. Cả 385 độ lệch LTM đều bằng +1 trong output. So với mốc 2026-06-29 dùng DEMO, số review sau mốc là 3, cặp trùng là 3 và lệch LTM là 2; đây là so sánh **hai nguồn khác phạm vi**, nên không diễn giải biến động của Santiago từ các hiệu số này.


## Khai báo sử dụng AI

- **Công cụ đã dùng:** Gemini; OpenAI Codex hỗ trợ điền phần nhận xét còn thiếu.
- **Yêu cầu chính:** tham số hóa năm phép kiểm QA cho mốc chụp 2025-09-27 và kiểm tra cửa sổ 12 tháng.
- **Kiểm chứng:** 27/09/2025 lùi một năm rồi cộng một ngày bằng 28/09/2024. Output của cell tự làm đã lưu: miền ngày 13/11/2010–27/09/2025, `review_sau_moc=0`, `review_mo_coi=0`, `trung_theo_cap=2378`, `lech_tong_review=0`, `lech_ltm=385`; 385 độ lệch đều bằng +1.
- **Giới hạn:** bảng đối chiếu cuối cell ghép mốc 2025 thật với mốc 2026 DEMO, nên không dùng để kết luận biến động giữa hai snapshot thật.


---

## 🧭 Hỗ trợ bài tập lớn (~30 phút — làm việc theo nhóm, không tính điểm lab)

Tự soát theo checklist; đánh dấu ✅/❌ và ghi một dòng "việc tuần tới" cho mỗi ❌:

1. ☐ Repo GitHub **riêng tư** đã mời tài khoản giảng viên và giảng viên thực hành
   (danh sách trên Canvas Portal); cấu trúc thư mục đúng theo đề.
2. ☐ **Mã tải dữ liệu** chạy được: tải đủ các mốc chụp bắt buộc của thành phố nhóm
   vào `data/raw/` — chưa commit dữ liệu thô lên repo.
3. ☐ **Kế hoạch phân tích** (bên đặt hàng, câu hỏi, KPI dự kiến, kế hoạch LLM) đã viết ra,
   cả nhóm thống nhất và chia thành đầu việc có người phụ trách.
4. ☐ **Bộ quy tắc QA nháp** (theo bài 10 + lab này): mỗi quy tắc đủ 4 phần
   tên–điều kiện–lý do–hành động; chạy thử trên mốc chụp mới nhất và tạo `qa_report` đầu tiên.
5. ☐ Phân công tuần tới: ai phụ trách **hợp phần LLM** (bài 11) — người đó tạo API key
   Gemini trước ở nhà (aistudio.google.com, miễn phí).

> Nhóm hoàn thành sớm có thể chạy bộ QA trên **một mốc chụp chưa dùng** để kiểm tra
> khả năng áp dụng trên mốc chụp lạ.

## Tóm tắt bài lab

| Bạn đã làm | Dùng cho |
|---|---|
| Kiểm miền thời gian và khoá ngoại | `qa_report` của bài tập lớn |
| Chứng minh bảng không có khoá tự nhiên | tránh dùng `drop_duplicates` máy móc |
| Đối chiếu cột dẫn xuất với số tự đếm | kiểm tra định nghĩa trước khi sử dụng |
| `qa_report.csv` từ các hàm nhận bảng qua tham số | chạy lại trên mốc chụp khác |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

Bài giảng tới: **mô hình ngôn ngữ lớn (LLM) cho dữ liệu phi cấu trúc**.
Hãy chuẩn bị API key Gemini theo hướng dẫn ở đầu notebook Bài 11.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** Nguyễn Tiến Long An / 25022763 / DSE2049_3

1. Giữ tên file **`lab-10.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_10.ipynb`).
2. Hoàn thành hàm TODO, các diễn giải và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
   Không commit `qa_report_*.csv` do notebook tạo ra.
4. Tạo tag `submit/lab-10/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-10` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [27]:
show_public_summary()

PUBLIC CHECKS — không phải điểm chính thức
Q1 · ví dụ công khai: ĐẠT
Q2 · ví dụ công khai: ĐẠT
Q3 · ví dụ công khai: ĐẠT
Q4 · ví dụ công khai: ĐẠT
Q5 · ví dụ công khai: ĐẠT
Q6 · ví dụ công khai: ĐẠT
Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.
